## Finding Similar structures with different melting points:

In [1]:
import os
import rootutils

from tqdm.notebook import tqdm

rootutils.setup_root(os.path.abspath('./'), indicator=".project-root", pythonpath=True, dotenv=True, cwd=True)

# auto-loading of imports from outside scripts
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from tqdm.auto import tqdm
import numpy as np

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.DataStructs import TanimotoSimilarity, BulkTanimotoSimilarity

# Some docs here: https://www.rdkit.org/docs/GettingStartedInPython.html

In [3]:
df = pd.read_csv("data_cod/coord_numbs_temp_smiles.csv")

### Remove bugged smiles (Tanimoto Dist breaks on them):
- [N-]=[N+]=N[P](N=[N+]=[N-])(N=[N+]=[N-])(N=[N+]=[N-])(N=[N+]=[N-])N=[N+]=[N-].c1ccc(P(=N[P+](c2ccccc2)(c2ccccc2)c2ccccc2)(c2ccccc2)c2ccccc2)cc1

In [4]:
df = df[~df["can_smiles"].isin(["[N-]=[N+]=N[P](N=[N+]=[N-])(N=[N+]=[N-])(N=[N+]=[N-])(N=[N+]=[N-])N=[N+]=[N-].c1ccc(P(=N[P+](c2ccccc2)(c2ccccc2)c2ccccc2)(c2ccccc2)c2ccccc2)cc1"])]

---

## Compute TanimotoDistance Matrix:

In [5]:
def compute_morgan_fps(smiles: str, radius: int = 2, nbits: int = 2048):
    try:
        mol = Chem.MolFromSmiles(smiles)
        mfpgen = Chem.rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=nbits)
        
        return mfpgen.GetFingerprint(mol)
    except Exception as e:
        with open('error_smiles.txt', 'a') as f:
            f.write(f"Error in fingerprints: {smiles}\n{str(e)}\n")
        return None

In [6]:
smiles_list = df["can_smiles"].tolist()

fps = [compute_morgan_fps(smiles) for smiles in tqdm(smiles_list, desc="Computing fingerprints")]

Computing fingerprints:   0%|          | 0/5586 [00:00<?, ?it/s]

In [7]:
def compute_similarity_matrix(fps):
    n = len(fps)
    M = np.zeros((n, n), dtype=float)
    for i in tqdm(range(n), desc="Computing Tanimoto matrix"):
        # compute similarities of fps[i] against fps[i+1:]
        sims = BulkTanimotoSimilarity(fps[i], fps[i+1:])
        # fill upper triangle and mirror to lower
        M[i, i+1:] = sims
        M[i+1:, i] = sims
        M[i, i] = 1.0  # similarity with self
    return M

In [13]:
sim_matrix = compute_similarity_matrix(fps)

Computing Tanimoto matrix: 100%|██████████| 5586/5586 [00:03<00:00, 1803.51it/s]
